# Aprendizado de Máquina — Lista prática 09

## Máquinas de Vetores de Suporte (SVM)

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

A SVM tem dois hiperparâmetros que interagem — $C$ e $\gamma$ — e uma grade
bidimensional é a primeira do curso. O resultado dela, neste banco, é um daqueles
que valem mais do que a resposta certa:

> **o melhor RBF encontrado pela busca é o que mais se parece com um kernel
> linear. E o kernel linear, sozinho, chega quase lá — em uma fração do tempo.**

Cada lacuna está marcada com `...`. Substitua **cada uma** pela sua resposta e
rode a célula.

---
## 1. Importando os pacotes

In [ ]:
import time

import numpy as np
from matplotlib.pyplot import subplots

import sklearn.model_selection as skm
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC

import warnings
warnings.filterwarnings("ignore")

---
## Exercício 1 — o que $C$ faz com a margem

Duas nuvens gaussianas que se sobrepõem um pouco — não são linearmente
separáveis, então a margem rígida do Exercício 1 da Lista Teórica 10 nem
existiria aqui.

Ajuste a SVM linear com três valores de $C$ e observe **três** quantidades: o
número de vetores de suporte, a largura da margem $1/\lVert w\rVert$, e a
acurácia de treino.

In [ ]:
rng = np.random.default_rng(2026)
n = 60

X = np.vstack([rng.normal([-1.5, 0], 0.8, size=(n // 2, 2)),
               rng.normal([1.5, 0], 0.8, size=(n // 2, 2))])
y = np.r_[-np.ones(n // 2), np.ones(n // 2)]

print("        C     vetores de suporte    margem   acuracia de treino")
for c in (0.01, 1.0, 100.0):
    modelo = SVC(kernel=..., C=...).fit(X, y)               # (a) e (b)
    margem = ...                 # (c)
    print(f"  {c:7.2f}          {modelo.n_support_.sum():2d}/{n}        "
          f"{margem:.4f}        {modelo.score(X, y):.4f}")

> **Sua vez.** Desenhe as duas nuvens e, por cima, a reta de decisão e as duas
> retas da margem, para $C=0{,}01$ e $C=100$. Marque os vetores de suporte com um
> círculo. (`modelo.support_vectors_` devolve as coordenadas.)

---
## Exercício 2 — a grade bidimensional

No kernel RBF, $K(x,z) = \exp(-\gamma\lVert x-z\rVert^2)$, os dois
hiperparâmetros fazem coisas diferentes: $C$ controla quanto se tolera errar, e
$\gamma$ controla o **alcance** de cada observação. Eles interagem, então é
preciso buscá-los juntos.

Padronizar antes é obrigatório: $\gamma$ multiplica uma distância euclidiana, e
sem padronizar a covariável de maior escala domina a soma.

In [ ]:
dados = load_breast_cancer()
X_bc, y_bc = dados.data, dados.target

tubo = Pipeline([("escala", StandardScaler()), ("svm", SVC(kernel="rbf"))])

grade = {
    "svm__C":     ...,                         # (a) de 0,01 a 1000
    "svm__gamma": ...,                         # (b) de 0,0001 a 10
}

cv = skm.StratifiedKFold(5, shuffle=True, random_state=2026)
busca = skm.GridSearchCV(tubo, grade, cv=cv, scoring="accuracy", n_jobs=-1).fit(X_bc, y_bc)

print(f"melhor: C = {busca.best_params_['svm__C']:g}, "
      f"gamma = {busca.best_params_['svm__gamma']:g}")
print(f"acuracia (CV) = {busca.best_score_:.4f}")

In [ ]:
tabela = busca.cv_results_["mean_test_score"].reshape(..., ...)     # (a) e (b) C x gamma

print("linhas = C, colunas = gamma")
print("        " + "".join(f"{g:9.4g}" for g in grade["svm__gamma"]))
for c, linha in zip(grade["svm__C"], tabela):
    print(f"  {c:7.4g} " + "".join(f"{v:9.4f}" for v in linha))

Vale conferir essa última leitura diretamente.

In [ ]:
linear = Pipeline([("escala", StandardScaler()), ("svm", SVC(kernel=...))])   # (a)
acc_linear = skm.cross_val_score(linear, X_bc, y_bc, cv=cv).mean()

print(f"RBF, melhor da grade : {busca.best_score_:.4f}")
print(f"kernel linear (C=1)  : {acc_linear:.4f}")

---
## Exercício 3 — o custo, medido

A SVM com kernel resolve um problema quadrático cuja matriz de Gram é
$n\times n$: a teoria promete custo entre $O(n^2)$ e $O(n^3)$. Já o `LinearSVC`
resolve o problema **primal**, com custo linear em $n$.

Meça os dois, e estime o expoente por regressão nos logaritmos.

In [ ]:
ns = np.array([1000, 2000, 4000, 8000, 16000])
tempo_svc, tempo_lin = [], []

for tamanho in ns:
    Xg, yg = make_classification(n_samples=int(tamanho), n_features=20,
                                 n_informative=10, random_state=0)
    t0 = time.time()
    SVC(kernel="rbf").fit(Xg, yg)
    tempo_svc.append(...)                           # (a)

    t0 = time.time()
    LinearSVC(max_iter=5000, dual=True).fit(Xg, yg)
    tempo_lin.append(time.time() - t0)

tempo_svc, tempo_lin = np.array(tempo_svc), np.array(tempo_lin)

print("      n      SVC(rbf)   LinearSVC")
for tamanho, a, b in zip(ns, tempo_svc, tempo_lin):
    print(f"  {tamanho:6d}   {a:8.3f}s  {b:8.3f}s")

print(f"\nexpoente SVC       : {np.polyfit(..., ..., 1)[0]:+.3f}")   # (b) e (c)
print(f"expoente LinearSVC : {np.polyfit(np.log(ns), np.log(tempo_lin), 1)[0]:+.3f}")

> **Sua vez.** Acrescente $n = 32\,000$ à lista e refaça o ajuste do expoente. Ele
> sobe ou desce? O que isso sugere sobre o regime em que a implementação está?

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 1 | de $C=0{,}01$ a $C=100$: vetores de suporte caem de 44 para 5, margem de 2,08 para 0,29 |
| 1 | a acurácia de treino **não** é monótona em $C$ — a SVM não minimiza o erro $0$--$1$ |
| 2 | com $\gamma\ge 1$ a acurácia trava em 0,6274, que é a prevalência da classe majoritária |
| 2 | o melhor RBF ($\gamma=0{,}0001$) é o que mais se parece com um linear — e o linear sozinho dá 0,9719 |
| 3 | expoentes medidos: `SVC` $+1{,}60$, `LinearSVC` $+1{,}07$ |
| 3 | ainda assim o `LinearSVC` é **mais lento** em todos os $n$ testados; só compensa perto de $n=200\,000$ |

**A seguir.** A Aula 10 fecha o bloco de classificação trazendo o KNN e as
árvores para cá, e põe todas as fronteiras do bloco lado a lado nos mesmos dados.